In [ ]:
from mienc import NonLinearEstimator as NLE
import numpy as np
from scipy.io import loadmat
import multiprocessing as mp
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.stats import chi2

mat = loadmat("../NonLinearityData/boldEso190.mat")["subj_tcs"]
newSIze = 3

In [ ]:
nle = NLE(config_file="config.ini", bins=8, surrogates=99, cache="cache", save_out=False, workers=23)
print(nle.estimate(mat[:, 25:28, 15:16]))

In [ ]:
_est = []
nle = NLE(config_file="config.ini", bins=8, surrogates=99, cache="cache", save_out=False, workers=1)
print(nle.estimate(mat[:, 50:53, 15:16]))
with mp.Pool(80) as pool:
    for e in tqdm(pool.imap_unordered(nle.estimate, (mat[:, 25:28, 15:16] for __ in range(10000))), total=10000):
        RNL = 1-e["gaussMI"]/e["totalMI"]
        sigma = e["sigmaGaussMI"]/e["totalMI"]
        _est.append([RNL, sigma])


In [ ]:
est = np.array(_est[:10000]).squeeze()
old_est = est.copy()
est[:,1]/=np.sqrt(99)
mean, est_std_mean = np.mean(est,0)
exp_std, est_std_std = np.std(est,0)

In [ ]:
plt.hist(est[:,0], bins="auto", density=True)
plt.vlines(mean, 1, 50, "r", "solid", label=r"$\mu$")
plt.vlines([mean-est_std_mean,mean+est_std_mean], 1, 50, "r", "dashed", label=r"$\langle\sigma^{EST}\rangle$")
plt.vlines([mean-exp_std,mean+exp_std], 1, 50, "k", "dotted", label=r"$\sigma^{EXP}$")
plt.plot(np.linspace(min(est[:,0]), max(est[:,0]),100), norm.pdf(np.linspace(min(est[:,0]), max(est[:,0]),100), mean, exp_std))
plt.legend(title="3 pairs")
plt.show()

In [ ]:
lower = np.sqrt(9999*exp_std**2/chi2.ppf(0.025, 9999))
upper = np.sqrt(9999*exp_std**2/chi2.ppf(0.975, 9999))
lo68, hi68 = 0.00140543, 0.00163722
lo95, hi95 = 0.00130265, 0.00175198
lo68, hi68, lo95, hi95 = np.sort(est[:,1])[[1587,8413, 250, 9750]]
plt.hist(est[:,1], bins="auto", density=True)
plt.vlines(est_std_mean, 1, 500, "r", "solid", label=r"$\langle\sigma^{EST}\rangle$")
plt.vlines([(est_std_mean-est_std_std),(est_std_mean+est_std_std)], 1, 500, "r", "dashed", label=r"$\sigma_{\sigma^{EST}}$")
plt.vlines([lo68, hi68], 1, 500, "y", "dotted", label=r"$\sigma^{EST}$ 68% ci")
plt.vlines([lo95, hi95], 1, 500, "g", "dotted", label=r"$\sigma^{EST}$ 95% ci", zorder=10)
plt.vlines(exp_std, 1, 500, "k", "solid", label=r"$\sigma^{EXP}$")
plt.fill_between([lower, upper], 1,500, color="gray", alpha=0.7, label=r"$\sigma^{EXP}$ 96% ci", zorder=3)
plt.plot(np.linspace(min(est[:,1]), max(est[:,1]),100), norm.pdf(np.linspace(min(est[:,1]), max(est[:,1]),100), est_std_mean, est_std_std))
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.load("../NonLinearityResultsNew/spiking_794812542_8000_bin6/correction_225_6.npy")
y = -0.5 * np.log(1 - (np.arange(200) / 200) ** 2)
plt.plot(x[:20], y[:20], ".-")

In [ ]:
from mienc import Corrector

c = Corrector(10,1000,200,50000,1000,None,None,24,False,False,False,"config.ini",True)
c.compute_correction()

In [ ]:
non_mono = c.correction
true_value = -0.5 * np.log(1 - (np.arange(200) / 200) ** 2)
last_decreasing = np.argmax(np.cumsum(np.diff(non_mono) < 0))
fit_order = 1 if last_decreasing < 10 else 2
fit = np.polyfit(true_value[:last_decreasing+3], non_mono[:last_decreasing+3], fit_order)

mono = non_mono.copy()
mono[:last_decreasing +
                3] = np.polyval(fit, true_value[:last_decreasing+3])

plt.plot(
    non_mono[:last_decreasing+3], true_value[:last_decreasing+3], ".-", label="Original", lw=1)
plt.plot(
    mono[:last_decreasing+3], true_value[:last_decreasing+3], ".-", label="Monotonic", lw=1)
plt.ylabel("True MI")
plt.xlabel("Estimated MI")
plt.legend()
plt.show()


In [ ]:
c.correction = non_mono
non_mono_corretti = c.correct(np.linspace(0.041,0.044,200))
c.correction = mono
mono_corretti = c.correct(np.linspace(0.041,0.044,200))
plt.plot(np.linspace(0.04,0.044,200), non_mono_corretti, ".-", label="Original")
plt.plot(np.linspace(0.04,0.044,200), mono_corretti, ".-", label="Monotonic")
plt.ylabel("True MI")
plt.xlabel("Estimated MI")
plt.legend()
plt.show()
plt.show()